In [ ]:
# ============================================================
# NUCLEAR THERMAL ROCKET (NTR)
# PSEUDO-2D LAVAL NOZZLE VISUALIZATION
# ============================================================
#
# Educational notebook
#
# Physics:
# - Quasi-1D compressible flow
# - Area-Mach relation
# - Isentropic expansion
# - Hydrogen propellant
#
# Visualizations:
# - Nozzle geometry
# - Mach contours
# - Pressure contours
# - Temperature contours
# - Streamlines
#
# Properties are shown ONLY INSIDE the nozzle.
#
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve

plt.rcParams["figure.figsize"] = (14,10)

# ============================================================
# ENGINE PARAMETERS
# ============================================================

gamma = 1.4

R = 4124.0                # Hydrogen [J/kg/K]

T0 = 2800.0               # Chamber temperature [K]

P0 = 7.0e6                # Chamber pressure [Pa]

# ============================================================
# NOZZLE GEOMETRY
# ============================================================

L = 4.0

x_throat = 1.5

Nx = 450
Ny = 250

x = np.linspace(0,L,Nx)

def nozzle_radius(x):

    Rin = 0.90
    Rt  = 0.35
    Rex = 1.00

    r = np.zeros_like(x)

    for i,xi in enumerate(x):

        if xi <= x_throat:

            r[i] = Rin - (Rin-Rt)*(xi/x_throat)**1.2

        else:

            s = (xi-x_throat)/(L-x_throat)

            r[i] = Rt + (Rex-Rt)*(s**1.4)

    return r

Rwall = nozzle_radius(x)

# ============================================================
# AREA DISTRIBUTION
# ============================================================

Area = np.pi*Rwall**2

Astar = np.min(Area)

AreaRatio = Area/Astar

# ============================================================
# AREA-MACH RELATION
# ============================================================

def area_mach(M,AR,gamma):

    term1 = 1.0/M

    term2 = (
        (2/(gamma+1))
        *
        (1 + (gamma-1)/2*M**2)
    )**((gamma+1)/(2*(gamma-1)))

    return term1*term2 - AR

Mach1D = np.zeros(Nx)

for i,AR in enumerate(AreaRatio):

    if x[i] < x_throat:
        guess = 0.2
    else:
        guess = 2.5

    Mach1D[i] = fsolve(
        area_mach,
        guess,
        args=(AR,gamma)
    )[0]

# ============================================================
# ISENTROPIC FLOW
# ============================================================

T1D = T0 / (
    1 + (gamma-1)/2*Mach1D**2
)

P1D = P0 * (
    T1D/T0
)**(gamma/(gamma-1))

rho1D = P1D/(R*T1D)

a1D = np.sqrt(gamma*R*T1D)

V1D = Mach1D*a1D

# ============================================================
# 2D GRID
# ============================================================

y = np.linspace(-1.05,1.05,Ny)

X,Y = np.meshgrid(x,y)

# ============================================================
# FLOW FIELDS
# ============================================================

Mach = np.full_like(X,np.nan)

Pressure = np.full_like(X,np.nan)

Temperature = np.full_like(X,np.nan)

Velocity = np.full_like(X,np.nan)

Ux = np.zeros_like(X)

Uy = np.zeros_like(X)

# ============================================================
# PSEUDO-2D RECONSTRUCTION
# ============================================================

for i in range(Nx):

    radius = Rwall[i]

    for j in range(Ny):

        r = abs(Y[j,i])

        if r <= radius:

            eta = r/radius

            profile = 1.0 - 0.08*eta**2

            M = Mach1D[i]*profile

            T = T0/(1+(gamma-1)/2*M**2)

            P = P0*(T/T0)**(gamma/(gamma-1))

            a = np.sqrt(gamma*R*T)

            V = M*a

            Mach[j,i] = M
            Temperature[j,i] = T
            Pressure[j,i] = P
            Velocity[j,i] = V

            Ux[j,i] = V

            Uy[j,i] = (
                0.03
                *V
                *Y[j,i]
                /radius
            )

# ============================================================
# FIGURE 1
# NOZZLE GEOMETRY
# ============================================================

plt.figure(figsize=(10,4))

plt.plot(x,Rwall,'k',lw=3)

plt.plot(x,-Rwall,'k',lw=3)

plt.fill_between(
    x,
    -Rwall,
    Rwall,
    color='lightgray'
)

plt.xlabel('Axial Position')

plt.ylabel('Radius')

plt.title(
    'Nuclear Thermal Rocket Laval Nozzle'
)

plt.grid(True)

plt.show()

# ============================================================
# FIGURE 2
# MACH CONTOURS
# ============================================================

plt.figure(figsize=(12,5))

cont = plt.contourf(
    X,
    Y,
    Mach,
    levels=50
)

plt.colorbar(
    cont,
    label='Mach Number'
)

plt.plot(x,Rwall,'k',lw=2)

plt.plot(x,-Rwall,'k',lw=2)

plt.xlabel('Axial Position')

plt.ylabel('Radial Position')

plt.title('Mach Number Distribution')

plt.tight_layout()

plt.show()

# ============================================================
# FIGURE 3
# PRESSURE CONTOURS
# ============================================================

plt.figure(figsize=(12,5))

cont = plt.contourf(
    X,
    Y,
    Pressure/1e6,
    levels=50
)

plt.colorbar(
    cont,
    label='Pressure [MPa]'
)

plt.plot(x,Rwall,'k',lw=2)

plt.plot(x,-Rwall,'k',lw=2)

plt.xlabel('Axial Position')

plt.ylabel('Radial Position')

plt.title('Pressure Distribution')

plt.tight_layout()

plt.show()

# ============================================================
# FIGURE 4
# TEMPERATURE CONTOURS
# ============================================================

plt.figure(figsize=(12,5))

cont = plt.contourf(
    X,
    Y,
    Temperature,
    levels=50
)

plt.colorbar(
    cont,
    label='Temperature [K]'
)

plt.plot(x,Rwall,'k',lw=2)

plt.plot(x,-Rwall,'k',lw=2)

plt.xlabel('Axial Position')

plt.ylabel('Radial Position')

plt.title('Temperature Distribution')

plt.tight_layout()

plt.show()

# ============================================================
# FIGURE 5
# STREAMLINES
# ============================================================

plt.figure(figsize=(12,5))

speed = np.sqrt(Ux**2 + Uy**2)

plt.streamplot(
    X,
    Y,
    Ux,
    Uy,
    density=2.0,
    color=speed
)

plt.plot(x,Rwall,'k',lw=2)

plt.plot(x,-Rwall,'k',lw=2)

plt.xlabel('Axial Position')

plt.ylabel('Radial Position')

plt.title('Pseudo-2D Streamlines')

plt.tight_layout()

plt.show()

# ============================================================
# PERFORMANCE SUMMARY
# ============================================================

Mexit = Mach1D[-1]

Texit = T1D[-1]

Pexit = P1D[-1]

Vexit = V1D[-1]

Isp = Vexit/9.81

print()
print("======================================")
print("NUCLEAR THERMAL ROCKET RESULTS")
print("======================================")
print(f"Exit Mach Number      : {Mexit:.3f}")
print(f"Exit Temperature [K]  : {Texit:.1f}")
print(f"Exit Pressure [MPa]   : {Pexit/1e6:.3f}")
print(f"Exit Velocity [m/s]   : {Vexit:.1f}")
print(f"Specific Impulse [s]  : {Isp:.1f}")
print("======================================")